# Notebook 03 — Kaplan-Meier Curves
**E. coli Levofloxacin — 2-State (S vs notS)**

Replicates `script6/03_ecoli_lev_2state_km.R`  
Validation targets:
- `output6/03_ecoli_lev_2state_km_summary_table.csv`
- R values: S→notS median = 3594d, notS→S median = 772d

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from pathlib import Path

R_OUT_DIR = Path('c:/ARMD/output6')
OUT_DIR   = Path('c:/ARMD/Python_Coursework_ARMD_analysis/python_output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 1. Load Covariate Dataset & Build 2-State

In [ ]:
cov = pd.read_csv(R_OUT_DIR / '03_ecoli_lev_cov_dataset.csv',
                  parse_dates=['order_time'])

if 'era_post2015' in cov.columns:
    cov = cov.drop(columns=['era_post2015'])

cov['state_2'] = cov['susceptibility'].apply(
    lambda x: 'S' if x == 'Susceptible' else 'notS'
)
cov = cov.sort_values(['anon_id', 'order_time']).reset_index(drop=True)

print(f'Rows: {len(cov):,}  |  Patients: {cov["anon_id"].nunique():,}')

## 2. Build Episode Dataset
An episode = a run of consecutive same-state observations for a patient.  
Duration = time from episode start to exit (or censoring at last observation).

In [ ]:
# Assign run_id: increments each time state changes within a patient
cov['state_change'] = (
    (cov['state_2'] != cov.groupby('anon_id')['state_2'].shift(1))
    .fillna(True)  # first row of each patient starts a new episode
)
cov['run_id'] = cov.groupby('anon_id')['state_change'].cumsum()

# Aggregate to episode level
ep = (
    cov.groupby(['anon_id', 'run_id'])
       .agg(
           ep_state    = ('state_2',    'first'),
           ep_start    = ('order_time', 'min'),
           ep_last_obs = ('order_time', 'max'),
           fq_at_entry = ('fq_90d',     'first'),
           tmx_at_entry= ('tmx_90d',    'first'),
       )
       .reset_index()
)

# Next episode state and start time (for determining events)
ep = ep.sort_values(['anon_id', 'run_id'])
ep['next_state'] = ep.groupby('anon_id')['ep_state'].shift(-1)
ep['next_time']  = ep.groupby('anon_id')['ep_start'].shift(-1)

# Duration: to next episode start if transitioning, else to last obs (censored)
ep['duration'] = np.where(
    ep['next_state'].notna(),
    (ep['next_time'] - ep['ep_start']).dt.total_seconds() / 86400,
    (ep['ep_last_obs'] - ep['ep_start']).dt.total_seconds() / 86400
)

ep['event'] = np.where(ep['next_state'].notna(), ep['next_state'], 'censored')

# Drop episodes with 0 or negative duration
ep = ep[ep['duration'] > 0].copy()

print(f'Total episodes: {len(ep):,}')
print(ep.groupby('ep_state')['event'].value_counts())

ep.to_csv(OUT_DIR / '03_ecoli_lev_km_episodes.csv', index=False)

## 3. Helper Functions

In [ ]:
def fit_km(episodes, origin, target):
    """Fit KM for origin->target transition. All other exits are censored."""
    sub = episodes[episodes['ep_state'] == origin].copy()
    sub['km_event'] = (sub['event'] == target).astype(int)
    kmf = KaplanMeierFitter()
    kmf.fit(sub['duration'], event_observed=sub['km_event'])
    return kmf, sub


def km_summary(kmf, sub, origin, target, label=''):
    """Extract summary stats from a fitted KM."""
    n_ep     = len(sub)
    n_events = sub['km_event'].sum()
    median   = kmf.median_survival_time_

    # 1-year probability (1 - S(365))
    timeline = kmf.survival_function_.index
    if 365 <= timeline.max():
        s1yr = float(kmf.survival_function_at_times(365).values[0])
        prob_1yr = round(1 - s1yr, 3)
    else:
        prob_1yr = None

    return {
        'cohort':             label or f'{origin}->{target}',
        'transition':         f'{origin}->{target}',
        'n_episodes':         n_ep,
        'n_events':           int(n_events),
        'median_time_days':   round(median, 1) if not np.isinf(median) else None,
        'prob_1yr_transition': prob_1yr,
    }


def plot_km_panel(ax, kmf_list, labels, colors, origin, target,
                  grey_threshold=10):
    """Plot one KM panel with CI shading and grey-out where n<threshold."""
    for kmf, label, color in zip(kmf_list, labels, colors):
        sf = kmf.survival_function_
        ci = kmf.confidence_interval_survival_function_
        t  = sf.index

        ax.step(t, sf.iloc[:, 0], where='post', color=color, label=label, lw=1.8)
        ax.fill_between(t, ci.iloc[:, 0], ci.iloc[:, 1],
                        step='post', alpha=0.15, color=color)

    ax.set_xlabel('Days')
    ax.set_ylabel('Probability of remaining in state')
    ax.set_title(f'{origin} → {target}')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

print('Helper functions defined.')

## 4. Figure 1 — Unstratified KM (1×2 panels)

In [ ]:
kmf_s2ns, sub_s2ns = fit_km(ep, 'S',    'notS')
kmf_ns2s, sub_ns2s = fit_km(ep, 'notS', 'S')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_km_panel(axes[0], [kmf_s2ns], ['S → notS'], ['#d73027'], 'S', 'notS')
plot_km_panel(axes[1], [kmf_ns2s], ['notS → S'], ['#2166ac'], 'notS', 'S')

# Annotate medians
for ax, kmf, label in [
    (axes[0], kmf_s2ns, 'S→notS'),
    (axes[1], kmf_ns2s, 'notS→S'),
]:
    med = kmf.median_survival_time_
    if not np.isinf(med):
        ax.axhline(0.5, color='grey', linestyle=':', lw=0.8)
        ax.axvline(med, color='grey', linestyle=':', lw=0.8)
        ax.text(med, 0.52, f'Median\n{med:.0f}d', fontsize=8, color='grey')
    else:
        ax.text(0.6, 0.55, 'Median not reached', transform=ax.transAxes,
                fontsize=8, color='grey')

plt.suptitle('E. coli Levofloxacin — 2-State KM (Unstratified)', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / '03_ecoli_lev_km_unstratified.png', dpi=150)
plt.show()

## 5. Figure 2 — Stratified by Prior FQ (fq_90d)

In [ ]:
def plot_stratified(ep, strat_col, strat_labels, title_suffix, filename):
    """1x2 KM plot stratified by a binary column."""
    colors = ['#2166ac', '#d73027']  # unexposed, exposed
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, (origin, target) in zip(axes, [('S', 'notS'), ('notS', 'S')]):
        base  = ep[ep['ep_state'] == origin].copy()
        base['km_event'] = (base['event'] == target).astype(int)

        kmfs, labs = [], []
        for val, lab in strat_labels.items():
            grp = base[base[strat_col] == val]
            if len(grp) < 5:
                continue
            kmf = KaplanMeierFitter()
            kmf.fit(grp['duration'], event_observed=grp['km_event'], label=lab)
            kmfs.append(kmf)
            labs.append(lab)

        if len(kmfs) == 2:
            lr = logrank_test(
                base[base[strat_col]==0]['duration'],
                base[base[strat_col]==1]['duration'],
                event_observed_A=base[base[strat_col]==0]['km_event'],
                event_observed_B=base[base[strat_col]==1]['km_event'],
            )
            plab = f'Log-rank p={lr.p_value:.3f}' if lr.p_value >= 0.001 else 'Log-rank p<0.001'
        else:
            plab = ''

        plot_km_panel(ax, kmfs, labs, colors[:len(kmfs)], origin, target)
        if plab:
            ax.text(0.98, 0.98, plab, transform=ax.transAxes,
                    ha='right', va='top', fontsize=8, color='dimgrey')

    plt.suptitle(f'E. coli Levofloxacin — KM Stratified by {title_suffix}', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=150)
    plt.show()


# Figure 2: stratified by fq_90d
if 'fq_at_entry' in ep.columns:
    plot_stratified(ep, 'fq_at_entry',
                    {0: 'No prior FQ', 1: 'Prior FQ (90d)'},
                    'Prior FQ Exposure',
                    '03_ecoli_lev_km_stratified_fq90d.png')

## 6. Figure 3 — Stratified by Prior TMP-SMX (tmx_90d)

In [ ]:
if 'tmx_at_entry' in ep.columns:
    plot_stratified(ep, 'tmx_at_entry',
                    {0: 'No prior TMP-SMX', 1: 'Prior TMP-SMX (90d)'},
                    'Prior TMP-SMX Exposure',
                    '03_ecoli_lev_km_stratified_tmx90d.png')

## 7. KM Summary Table

In [ ]:
summary_rows = [
    km_summary(kmf_s2ns, sub_s2ns, 'S',    'notS', 'E. coli Levofloxacin (2-state)'),
    km_summary(kmf_ns2s, sub_ns2s, 'notS', 'S',    'E. coli Levofloxacin (2-state)'),
]

km_table = pd.DataFrame(summary_rows)
km_table.to_csv(OUT_DIR / '03_ecoli_lev_km_summary_table.csv', index=False)
print(km_table.to_string(index=False))

### Validation vs R

In [ ]:
r_km = pd.read_csv(R_OUT_DIR / '03_ecoli_lev_2state_km_summary_table.csv')
print('--- R KM summary ---')
print(r_km.to_string(index=False))
print()
print('--- Python KM summary ---')
print(km_table.to_string(index=False))
print()
print('Expected: S->notS median ~3594d,  notS->S median ~772d')